# DAX30 Distance-Correlation Network: Temporal Evolution

Extend the static single-window distance-correlation network into a time series of networks via rolling windows over trading-day observations. Track network-level metrics (connectivity, density, clustering) and per-node metrics (degree, centrality) to understand how DAX30 equity correlations evolved from 2007 to 2019.

## Imports

In [ ]:
from __future__ import annotations

import warnings
from collections.abc import Iterator
from dataclasses import dataclass
from datetime import date

import networkx as nx
import polars as pl
from plotnine import *
from tqdm.auto import tqdm

from tgraphportfolio.analysis import measures, network, transforms
from tgraphportfolio.backends.duckdb_backend import DuckDBSource

warnings.filterwarnings("ignore")

## Windowing Strategy: Rolling vs. Expanding vs. EWMA-Weighted

Three ways to turn a single static network into a time series of networks:

**Rolling (fixed-length sliding) window â€” chosen default.**
Each window covers exactly `window_size` consecutive trading-day observations;
the window slides forward by `step` observations each iteration. Every window
has the same sample size, so dcor estimates are comparable across time and the
network is equally sensitive to a regime change wherever it falls in the
window. Older observations are fully forgotten once they fall outside the
window â€” this is a feature (captures genuine regime change) and a limitation
(estimates are noisier for short windows, and abrupt full "vintage swap" of
the sample at each step can produce visible jumps in the network).

**Expanding window â€” a one-flag variant of the same code.**
The window start is pinned to the first available date; only the end advances.
Every window is a superset of the previous one. This smooths metric evolution
(more data â†’ lower variance) but means early history's influence never fully
decays â€” a correlation regime from 2007 keeps a residual footprint on a 2015
network, which understates how much reality has moved on. It also converges
in cost: the last window's dcor calls are as slow as a single full-history run.
Implemented here as `expanding=True` on the same `generate_windows` /
`compute_window_metrics` functions used for rolling â€” no separate code path.

**EWMA-weighted correlation â€” not implemented, flagged as future work.**
An exponentially-weighted rolling correlation (recent observations weighted
more heavily, no hard cutoff) is a common third option for standard Pearson
correlation (`pandas`/`polars` `.ewm().corr()`). Distance correlation has no
native weighted formulation â€” `dcor.distance_correlation` takes an
unweighted sample. Approximating it would require either (a) resampling
observations by weight (bootstrap/importance resampling proportional to EWMA
weights) before calling `dcor.distance_correlation`, or (b) computing dcor on
a weighted double-centered distance matrix by hand (bypassing the `dcor`
package's U-statistic estimator entirely, and losing its bias-corrected /
fast O(n log n) univariate path). Both are non-trivial, unvalidated
extensions of the underlying statistic, not a simple keyword flag â€” out of
scope for this notebook.

**Default chosen: rolling, `window_size` â‰ˆ 1 trading year, `step` â‰ˆ 1 trading
month.** See the "Pick sane defaults" cell below for the concrete numbers,
computed from this DB's actual DAX30 date range rather than hardcoded. Switch
to expanding by setting `EXPANDING = True` â€” no other code changes needed.

## Load DAX30 Data

In [ ]:
DUCKDB_PATH = r"D:\data\duckdb\equity_eod_data.duckdb"
TABLE = "equity_eod"
EQ_INDEX = "DAX30"
DATE_COL, NAME_COL, VALUE_COL = "Date", "Name", "Close"

with DuckDBSource(DUCKDB_PATH, read_only=True) as db:
    df = db.run_query(
        f"""
        SELECT CAST(\"Index\" AS DATE) AS Date, Stock AS Name, Close
        FROM {TABLE}
        WHERE EqIndex = '{EQ_INDEX}'
        ORDER BY Date, Name
        """
    )

df = df.with_columns(pl.col(DATE_COL).cast(pl.Date)).sort(DATE_COL, NAME_COL)

## Compute Data Bounds

In [ ]:
bounds = df.select(
    pl.col(DATE_COL).min().alias("min_date"),
    pl.col(DATE_COL).max().alias("max_date"),
)
n_nodes_raw = df.get_column(NAME_COL).n_unique()
n_dates_raw = df.get_column(DATE_COL).n_unique()
n_rows = df.height

print(
    f"{EQ_INDEX}: {n_dates_raw} dates ({bounds['min_date'][0]} to {bounds['max_date'][0]}), "
    f"{n_nodes_raw} stocks, {n_rows:,} rows"
)

## Compute Daily Returns

In [ ]:
df_returns = transforms.apply_transforms(
    df,
    transform_ids=["daily_returns"],
    date_column=DATE_COL,
    name_column=NAME_COL,
    value_columns=[VALUE_COL],
)
dates = df_returns.get_column(DATE_COL).unique().sort().to_list()
print(f"After daily_returns: {len(dates)} unique dates")

## Rolling/Expanding Window Generator

In [ ]:
def generate_windows(
    dates: list[date],
    window_size: int,
    step: int,
    *,
    expanding: bool = False,
) -> Iterator[tuple[date, date, list[date]]]:
    """Yield rolling or expanding windows over sorted unique trading dates.

    Rolling (expanding=False): each window is exactly `window_size`
    consecutive observations, advancing by `step` observations per
    iteration. Expanding (expanding=True): window start is pinned to
    dates[0]; only the end advances, starting once `window_size`
    observations are available.

    Args:
        dates: Sorted, unique trading dates (ascending).
        window_size: Minimum/initial number of observations per window.
        step: Number of observations to advance between windows.
        expanding: Anchor window start at dates[0] instead of sliding it.

    Yields:
        (window_start, window_end, window_dates) tuples; window_dates
        spans [window_start, window_end] inclusive.

    Raises:
        ValueError: If window_size < 3 (dcor needs >=3 paired obs).
    """
    if window_size < 3:
        raise ValueError("window_size must be >= 3 for dcor to be defined")
    n = len(dates)
    end_idx = window_size - 1
    while end_idx < n:
        start_idx = 0 if expanding else end_idx - window_size + 1
        window_dates = dates[start_idx : end_idx + 1]
        yield window_dates[0], window_dates[-1], window_dates
        end_idx += step

## Pick Sane Defaults

In [ ]:
@dataclass
class EvolutionConfig:
    """Parameters for one rolling/expanding-window evolution run.

    Plain dataclass rather than a pydantic model: this is a notebook-local,
    single-process config with no external/user input to validate, so
    pydantic's runtime validation is unneeded overhead here (would be
    worthwhile if this config were exposed via a CLI/API/GUI, as
    PipelineConfig's fields conceptually parallel).
    """

    window_size: int = 252       # ~1 trading year
    step: int = 21               # ~1 trading month
    expanding: bool = False
    min_nodes: int = 5
    independent_threshold: float = 0.33
    centrality: str = "eigenvector"


CFG = EvolutionConfig()

n_nodes = df_returns.get_column(NAME_COL).n_unique()
n_pairs_per_window = n_nodes * (n_nodes - 1) // 2
n_windows = sum(
    1
    for _ in generate_windows(
        dates, CFG.window_size, CFG.step, expanding=CFG.expanding
    )
)
total_dcor_calls = n_windows * n_pairs_per_window
print(
    f"{n_windows} windows Ã— {n_pairs_per_window} pairs/window = {total_dcor_calls:,} dcor calls"
)

## Performance Estimate

**Performance risk.** dcor cost is ~O(pairs) per window and pairs grows
O(nodesÂ²); the existing notebook's single full-history run (~1250 obs, ~93 nodes â†’ ~4371 pairs) took ~10s
on this machine â†’ ~440 pairs/sec. Rolling windows here use only ~252 obs (~20% of that),
so per-pair cost should be *at or below* that rate (dcor's default fast
univariate algorithm is roughly O(n log n) in sample size). Using 440
pairs/sec as a rough, non-conservative estimate:

- **Recommended default** (window=252, step=21): ~62,775 calls â†’ **~2â€“7 minutes**.
- **Weekly step** (window=252, step=5): 565 windows Ã— 465 â‰ˆ 262,725 calls â†’
  **~10â€“30 minutes** â€” avoid as a default, only for a final high-resolution
  pass once the pipeline is validated.
- **Expanding mode**: same window count for a given step, but later windows
  carry up to the full ~3073-obs sample, so the tail of the run is slower
  than the rolling equivalent â€” budget extra time versus the rolling case.

**Always smoke-test on a truncated date range first** (next cell) before
launching the full-history run.

## Per-Window Metric Helpers

In [ ]:
def _drop_nan_edges(graph: nx.Graph) -> nx.Graph:
    """Remove NaN-weight edges left behind by build_corr_nx.

    build_corr_nx's threshold check (`wt >= 1 - threshold`) is False for
    NaN weights (insufficient paired observations in measures.py), so such
    edges silently survive pruning. Handled defensively here rather than
    in analysis/network.py, which is out of scope for this notebook.
    """
    graph = graph.copy()
    bad_edges = [(u, v) for u, v, w in graph.edges(data="weight") if w != w]  # NaN check
    graph.remove_edges_from(bad_edges)
    return graph


def _add_strength_attr(graph: nx.Graph) -> None:
    """Attach a 'strength' edge attribute (= 1 - weight) in place.

    build_corr_nx stores dissimilarity (1 - dcor) as 'weight' for layout
    purposes. Degree/centrality that should track *strong correlation*
    (not distance) must use this derived attribute instead.
    """
    for _, _, d in graph.edges(data=True):
        d["strength"] = 1.0 - d["weight"]


def _node_centrality(graph: nx.Graph, centrality: str) -> dict[str, float]:
    """Compute one per-node centrality measure, robust to disconnected graphs.

    Args:
        graph: Window's pruned similarity graph (must already have 'strength').
        centrality: "eigenvector" (default, weight='strength'; falls back to
            betweenness on non-convergence), "betweenness" (weight='weight',
            i.e. dissimilarity-as-distance), or "degree" (unweighted).

    Returns:
        Mapping of node -> centrality value.
    """
    if graph.number_of_edges() == 0:
        return dict.fromkeys(graph.nodes(), 0.0)
    if centrality == "eigenvector":
        try:
            return nx.eigenvector_centrality(graph, weight="strength", max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            return nx.betweenness_centrality(graph, weight="weight")
    if centrality == "betweenness":
        return nx.betweenness_centrality(graph, weight="weight")
    if centrality == "degree":
        return nx.degree_centrality(graph)
    raise ValueError(f"Unknown centrality: {centrality!r}")


def _network_summary(graph: nx.Graph, window_start: date, window_end: date) -> dict:
    """One row of network-level summary metrics for a window's graph."""
    n_nodes = graph.number_of_nodes()
    degrees = [d for _, d in graph.degree()]
    largest_cc = max((len(c) for c in nx.connected_components(graph)), default=0)
    return {
        "window_start": window_start,
        "window_end": window_end,
        "n_nodes": n_nodes,
        "n_edges": graph.number_of_edges(),
        "density": nx.density(graph),
        "avg_degree": (sum(degrees) / n_nodes) if n_nodes else float("nan"),
        "n_components": nx.number_connected_components(graph) if n_nodes else 0,
        "largest_component_size": largest_cc,
        "avg_clustering": nx.average_clustering(graph) if n_nodes else float("nan"),
    }


def _node_summary(
    graph: nx.Graph, window_end: date, *, centrality: str
) -> list[dict]:
    """Long-format per-node metric rows (degree, weighted_degree, centrality) for one window."""
    cent = _node_centrality(graph, centrality)
    weighted_degree = dict(graph.degree(weight="strength"))
    rows = []
    for node in graph.nodes():
        rows.append(
            {
                "window_end": window_end,
                "node": node,
                "metric": "degree",
                "value": float(graph.degree(node)),
            }
        )
        rows.append(
            {
                "window_end": window_end,
                "node": node,
                "metric": "weighted_degree",
                "value": float(weighted_degree[node]),
            }
        )
        rows.append(
            {
                "window_end": window_end,
                "node": node,
                "metric": centrality,
                "value": float(cent.get(node, float("nan"))),
            }
        )
    return rows

## Main Loop: Compute Window Metrics

In [ ]:
def compute_window_metrics(
    df_returns: pl.DataFrame,
    dates: list[date],
    cfg: EvolutionConfig,
    *,
    date_column: str = DATE_COL,
    name_column: str = NAME_COL,
    value_column: str = VALUE_COL,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Compute network- and node-level metrics for every window.

    Args:
        df_returns: Long-format daily-returns dataframe.
        dates: Sorted unique trading dates present in df_returns.
        cfg: Windowing/measure/threshold parameters.
        date_column, name_column, value_column: Column names in df_returns.

    Returns:
        (network_metrics, node_metrics) Polars DataFrames: one row per
        window (network-level), and one row per (window_end, node, metric)
        (node-level, long format).
    """
    windows = list(
        generate_windows(
            dates, cfg.window_size, cfg.step, expanding=cfg.expanding
        )
    )
    network_rows, node_rows = [], []
    pbar = tqdm(windows, desc="windows")
    for window_start, window_end, window_dates in pbar:
        window_df = df_returns.filter(
            pl.col(date_column).is_between(window_start, window_end)
        )
        wide = network.pivot_to_wide(
            window_df, date_column, name_column, value_column
        )
        nodes = [c for c in wide.columns if c != date_column]
        nodes = [
            n
            for n in nodes
            if wide.get_column(n).drop_nulls().len() >= 3
        ]
        if len(nodes) < cfg.min_nodes:
            continue
        measure_df = measures.compute_measure(
            "distance_correlation", wide.select(nodes), nodes
        )
        graph = network.build_corr_nx(
            measure_df, independent_threshold=cfg.independent_threshold
        )
        graph = _drop_nan_edges(graph)
        _add_strength_attr(graph)
        network_rows.append(_network_summary(graph, window_start, window_end))
        node_rows.extend(
            _node_summary(graph, window_end, centrality=cfg.centrality)
        )
        pbar.set_postfix(window_end=str(window_end), n_edges=graph.number_of_edges())
    return pl.DataFrame(network_rows), pl.DataFrame(node_rows)

## Smoke Test

In [ ]:
SMOKE_END = date(2010, 1, 1)
df_smoke = df_returns.filter(pl.col(DATE_COL) < SMOKE_END)
dates_smoke = df_smoke.get_column(DATE_COL).unique().sort().to_list()

%time net_smoke, node_smoke = compute_window_metrics(df_smoke, dates_smoke, CFG)
print(f"Smoke test: {net_smoke.height} windows")

## Full Run

In [ ]:
%time network_metrics, node_metrics = compute_window_metrics(df_returns, dates, CFG)
print(f"Full run: {network_metrics.height} windows")

## Top Variable Nodes

In [ ]:
def top_variable_nodes(
    node_metrics: pl.DataFrame, metric: str, k: int = 6
) -> list[str]:
    """Return the k node names with the highest across-window std for `metric`."""
    return (
        node_metrics.filter(pl.col("metric") == metric)
        .group_by("node")
        .agg(pl.col("value").std().alias("std"))
        .sort("std", descending=True)
        .head(k)
        .get_column("node")
        .to_list()
    )

## Plot (a): Network-Level Metrics (Faceted Time Series)

In [ ]:
network_long = network_metrics.unpivot(
    index=["window_start", "window_end"],
    on=[
        "n_edges",
        "density",
        "avg_degree",
        "n_components",
        "largest_component_size",
        "avg_clustering",
    ],
    variable_name="metric",
    value_name="value",
)

(
    ggplot(network_long.to_pandas(), aes(x="window_end", y="value"))
    + geom_line(color="#2a78d6", size=0.7)
    + geom_point(color="#2a78d6", size=0.9, alpha=0.6)
    + facet_wrap("~metric", scales="free_y", ncol=2)
    + labs(
        x="Window end date",
        y=None,
        title="DAX30 distance-correlation network: rolling metrics",
    )
    + theme_minimal()
    + theme(figure_size=(10, 9))
)

## Plot (b): Node Ã— Time Heatmap (Weighted Degree)

In [ ]:
heatmap_df = node_metrics.filter(pl.col("metric") == "weighted_degree")

(
    ggplot(heatmap_df.to_pandas(), aes(x="window_end", y="node", fill="value"))
    + geom_tile()
    + scale_fill_gradient(low="#cde2fb", high="#0d366b", name="Weighted\ndegree")
    + labs(
        x="Window end date",
        y="Stock",
        title="DAX30 node weighted-degree evolution",
    )
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_y=element_text(size=6))
)

## Plot (c): Notable Nodes (Centrality Evolution)

In [ ]:
notable = top_variable_nodes(node_metrics, metric=CFG.centrality, k=6)
line_df = node_metrics.filter(
    (pl.col("metric") == CFG.centrality) & (pl.col("node").is_in(notable))
)

CATEGORICAL_6 = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
]

(
    ggplot(line_df.to_pandas(), aes(x="window_end", y="value", color="node"))
    + geom_line(size=0.8)
    + scale_color_manual(values=CATEGORICAL_6)
    + labs(
        x="Window end date",
        y=f"{CFG.centrality.title()} centrality",
        color="Stock",
        title="Most variable DAX30 nodes over time",
    )
    + theme_minimal()
    + theme(figure_size=(10, 6))
)

## Before Committing

**Clear all notebook outputs** (`Kernel > Restart & Clear Output` or `jupyter nbconvert --clear-output`) â€” this notebook produces large tqdm/plot outputs that shouldn't go into version control.

# Extended Analysis: Structural Statistics, Regime Detection, Communities & Trajectories

Four additional "chapters" built on `graspologic`, appended after the original rolling-window
analysis above (cells 0–31, untouched). Each chapter answers a question the original
`compute_window_metrics` pipeline can't, because that pipeline discards the per-window
`nx.Graph` objects after computing scalar summaries:

- **Chapter 0** (this section) reruns the windowing/dcor pipeline, this time keeping the graphs,
  and builds a consistent-node-order adjacency tensor.
- **Chapter A** — two more rolling descriptive statistics (global transitivity, giant-component
  average shortest path).
- **Chapter B** — regime/change-point detection via `graspologic.inference.latent_position_test`
  between consecutive windows, with a hand-rolled Holm-Bonferroni correction.
- **Chapter C** — per-window community detection (`AdjacencySpectralEmbed` + `KMeansCluster`)
  and drift tracking via Adjusted Rand Index.
- **Chapter D** — a single joint `OmnibusEmbed` across all windows, producing aligned node
  trajectories through latent space over time.

**Requires** [`graspologic`](https://github.com/FulgentMcGuffin/graspologic) from `pyproject.toml` (installed by `uv sync`).

## New Imports (graspologic)

Installed via `uv sync` from the [Python 3.13 / NumPy 2–compatible fork](https://github.com/FulgentMcGuffin/graspologic) (see `pyproject.toml`).

In [ ]:
import numpy as np
from graspologic.cluster import KMeansCluster
from graspologic.embed import AdjacencySpectralEmbed, OmnibusEmbed
from graspologic.inference import latent_position_test
from graspologic.utils import remap_labels
from sklearn.metrics import adjusted_rand_score

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

## Chapter 0: Graph-Preserving Rolling-Window Pass (Shared Foundation)

`compute_window_metrics` (above) only returns scalar summary rows — the `nx.Graph` built
for each window is discarded once `_network_summary`/`_node_summary` have been extracted
from it. Chapters B, C, and D below all need the actual graphs (or their adjacency matrices),
and specifically need them with a **consistent node set in a consistent order** across every
window, since `graspologic`'s multi-graph methods (`latent_position_test`, `OmnibusEmbed`)
assume identically-shaped, identically-ordered inputs.

This section reruns the windowing + dcor + graph-construction pipeline (reusing
`generate_windows`, `network.pivot_to_wide`, `measures.compute_measure`, `network.build_corr_nx`,
`_drop_nan_edges`, `_add_strength_attr` verbatim — none of these are reimplemented) but this time
**keeps** each window's graph instead of collapsing it to a summary row. Because this duplicates
the expensive dcor computation from scratch, it uses a coarser `step` than the original `CFG`
(see the new `CFG_GRAPHS` below) to keep the second pass affordable.

In [ ]:
def collect_window_graphs(
    df_returns: pl.DataFrame,
    dates: list[date],
    cfg: EvolutionConfig,
    *,
    date_column: str = DATE_COL,
    name_column: str = NAME_COL,
    value_column: str = VALUE_COL,
) -> tuple[dict[date, nx.Graph], dict[date, date]]:
    """Rerun the rolling-window dcor pipeline, keeping the pruned graph itself.

    Duplicates compute_window_metrics' windowing/measure/graph-construction calls
    (generate_windows, network.pivot_to_wide, measures.compute_measure,
    network.build_corr_nx, _drop_nan_edges, _add_strength_attr) but returns the
    nx.Graph per window instead of discarding it after computing scalar summaries --
    needed for graspologic's per-window-graph methods (Chapters B/C/D below), which
    compute_window_metrics does not preserve.

    Args:
        df_returns: Long-format daily-returns dataframe.
        dates: Sorted unique trading dates present in df_returns.
        cfg: Windowing/measure/threshold parameters. Recommend a coarser `step`
            than the main CFG (e.g. CFG_GRAPHS), since this repeats the dcor
            computation from scratch.
        date_column, name_column, value_column: Column names in df_returns.

    Returns:
        (graphs, window_starts): graphs maps window_end -> pruned, strength-annotated
        nx.Graph, insertion-ordered chronologically (window_end dates are strictly
        increasing, so no key collisions); window_starts maps the same window_end
        keys -> window_start, kept separately so _network_summary (which takes both
        dates) can be reused unmodified in Chapter A below.
    """
    windows = list(
        generate_windows(dates, cfg.window_size, cfg.step, expanding=cfg.expanding)
    )
    graphs: dict[date, nx.Graph] = {}
    window_starts: dict[date, date] = {}
    pbar = tqdm(windows, desc="windows (graph-preserving)")
    for window_start, window_end, window_dates in pbar:
        window_df = df_returns.filter(
            pl.col(date_column).is_between(window_start, window_end)
        )
        wide = network.pivot_to_wide(window_df, date_column, name_column, value_column)
        nodes = [c for c in wide.columns if c != date_column]
        nodes = [n for n in nodes if wide.get_column(n).drop_nulls().len() >= 3]
        if len(nodes) < cfg.min_nodes:
            continue
        measure_df = measures.compute_measure(
            "distance_correlation", wide.select(nodes), nodes
        )
        graph = network.build_corr_nx(
            measure_df, independent_threshold=cfg.independent_threshold
        )
        graph = _drop_nan_edges(graph)
        _add_strength_attr(graph)
        graphs[window_end] = graph
        window_starts[window_end] = window_start
        pbar.set_postfix(window_end=str(window_end), n_nodes=graph.number_of_nodes())
    return graphs, window_starts

## Chapter 0 — Config: Coarser Cadence for the Graph-Preserving Pass

In [ ]:
CFG_GRAPHS = EvolutionConfig(step=63)  # quarterly cadence (window unchanged at 252)

n_windows_graphs = sum(
    1
    for _ in generate_windows(
        dates, CFG_GRAPHS.window_size, CFG_GRAPHS.step, expanding=CFG_GRAPHS.expanding
    )
)
total_dcor_calls_graphs = n_windows_graphs * n_pairs_per_window
print(
    f"{n_windows_graphs} windows × {n_pairs_per_window} pairs/window = "
    f"{total_dcor_calls_graphs:,} dcor calls"
)

## Chapter 0 — Performance Estimate
Reuses `n_pairs_per_window` from the original "Pick Sane Defaults" cell above (unchanged,
since `CFG_GRAPHS` only changes `step`). At `step=63` (3× coarser than the original `step=21`),
the window count and total dcor-call count printed above should come out to roughly **1/3** of
the original full run's count, since the window count scales ~linearly with `1/step` for a
fixed date range. Applying the same ~440 pairs/sec rough rate used in the original Performance
Estimate cell to whatever `total_dcor_calls_graphs` printed above turns out to be: expect very
roughly **one third of the original run's ~2–7 minute estimate**, i.e. somewhere in the ~45 sec
– 2.5 min range on this machine — **this is a rough order-of-magnitude, not a benchmark**; the
exact call count is only known once the cell above has actually run.
**Always smoke-test on a truncated date range first** (next cell), reusing the same
`df_smoke`/`dates_smoke` truncation (pre-2010) from the original Smoke Test cell above.

In [ ]:
%time graphs_smoke, window_starts_smoke = collect_window_graphs(df_smoke, dates_smoke, CFG_GRAPHS)
print(f"Smoke test: {len(graphs_smoke)} window graphs collected")

## Chapter 0 — Full Run: Collect Window Graphs

In [ ]:
%time graphs_by_window, window_starts = collect_window_graphs(df_returns, dates, CFG_GRAPHS)
print(f"Full run: {len(graphs_by_window)} window graphs collected")

## Chapter 0 — Build Common-Node Adjacency Tensor
`graspologic`'s multi-graph methods (`latent_position_test`, `OmnibusEmbed`) require every
graph to share an identical node set in an identical order. The DAX30 panel is *expected* to
be fully balanced (all 30 constituents present in every window), but that must be **computed,
not assumed** — the `<3 non-null obs` node-drop in `collect_window_graphs` could in principle
remove different nodes in different windows. Edges are stored as **binary** (`weight=None`)
rather than continuous dissimilarity, matching the book's RDPG/SBM binary-adjacency assumption
used by `latent_position_test`/`ASE`/`OmnibusEmbed`.

In [ ]:
def build_adjacency_tensor(
    graphs: dict[date, nx.Graph],
) -> tuple[list[date], list[str], np.ndarray]:
    """Stack per-window graphs into a binary (T, n, n) adjacency tensor.

    Restricts to the intersection of node sets across all collected windows (computed,
    not assumed) and uses binary edges (weight=None) rather than the continuous
    dissimilarity 'weight' attribute.

    Args:
        graphs: window_end -> nx.Graph, as returned by collect_window_graphs.

    Returns:
        (window_ends, common_nodes, tensor): window_ends is the sorted list of
        window-end dates (chronological, tensor axis-0 order); common_nodes is the
        sorted node names shared by every graph (tensor axis-1/2 order); tensor has
        shape (T, n, n), dtype float64, values in {0.0, 1.0}.

    Raises:
        ValueError: If fewer than CFG.min_nodes nodes are common to every collected
            window.
    """
    window_ends = sorted(graphs.keys())
    node_sets = [set(graphs[we].nodes()) for we in window_ends]
    common_nodes = sorted(set.intersection(*node_sets))
    print(
        f"{len(window_ends)} windows; common node set: {len(common_nodes)} of "
        f"{max(len(s) for s in node_sets)} max nodes seen in any single window"
    )
    if len(common_nodes) < CFG.min_nodes:
        raise ValueError(
            f"Only {len(common_nodes)} common nodes across all windows -- "
            f"too few for the graspologic chapters below."
        )
    tensor = np.stack(
        [nx.to_numpy_array(graphs[we], nodelist=common_nodes, weight=None) for we in window_ends]
    )
    return window_ends, common_nodes, tensor

In [ ]:
window_ends, common_nodes, adj_tensor = build_adjacency_tensor(graphs_by_window)
DHAT = int(np.ceil(np.log2(len(common_nodes))))  # shared latent dim, reused by Ch. B & D
print(f"Adjacency tensor shape: {adj_tensor.shape}")
print(f"Shared latent dimension DHAT = {DHAT} (ceil(log2({len(common_nodes)})))")

## Chapter A: Extended Rolling Descriptive Statistics

`nx.transitivity` (global clustering — ratio of closed triplets to all triplets) is
distinct from the existing `avg_clustering` (`nx.average_clustering`, mean of *local*
clustering coefficients); giant-component `nx.average_shortest_path_length` (restricted
to the largest connected component, since the unrestricted call raises on disconnected
graphs; NaN if giant component has <2 nodes). Both unweighted, matching the existing
`avg_clustering` convention.

In [ ]:
def _extended_network_summary(graph: nx.Graph, window_end: date) -> dict:
    """Extra global structural metrics not covered by _network_summary.

    Args:
        graph: Pruned DAX30 window graph (undirected, unweighted).
        window_end: Window end date (for the returned dict key).

    Returns:
        Row dict with keys: window_end, transitivity, giant_component_avg_shortest_path.
        NaNs used for disconnected graphs (transitivity still defined, avg_shortest_path is NaN).
    """
    result = {"window_end": window_end}
    result["transitivity"] = nx.transitivity(graph) or np.nan
    largest_cc = max(nx.connected_components(graph), key=len, default=set())
    if len(largest_cc) >= 2:
        subgraph = graph.subgraph(largest_cc)
        result["giant_component_avg_shortest_path"] = nx.average_shortest_path_length(subgraph)
    else:
        result["giant_component_avg_shortest_path"] = np.nan
    return result

In [ ]:
extended_rows = []
for window_end in graphs_by_window.keys():
    graph = graphs_by_window[window_end]
    row = _extended_network_summary(graph, window_end)
    extended_rows.append(row)
extended_network_metrics = pl.DataFrame(extended_rows)
print(f"Extended network metrics: {extended_network_metrics.shape}")
extended_network_metrics.head()

In [ ]:
# Plot (d): Extended metrics (8 facets total, original 6 + transitivity + giant-comp ASP)

# ["window_start", "window_end", "n_nodes", "n_edges", "density", "avg_degree", "n_components", "largest_component_size", "avg_clustering", "transitivity", "giant_component_avg_shortest_path"]
metrics_to_plot = network_metrics.join(
    extended_network_metrics, on="window_end", how="inner"
).select(
    "window_end", "avg_degree", "avg_clustering", "density",
    "transitivity", "largest_component_size", 
    "giant_component_avg_shortest_path", "n_components", "n_edges", "n_nodes"
).unpivot(
    index="window_end", variable_name="metric", value_name="value"
)

(
    ggplot(metrics_to_plot, aes("window_end", "value"))
    + geom_line(size=0.8, color="#555555")
    + facet_wrap("~metric", ncol=4, scales="free_y")
    + labs(x="Window End", y="Metric Value", title="Plot (d): Extended Rolling Metrics")
    + theme_minimal()
    + theme(figure_size=(14, 6), subplots_adjust={"top": 0.92})
)

## Chapter B: Regime / Change-Point Detection

Statistical hypothesis testing between consecutive network snapshots via
`graspologic.inference.latent_position_test`, adapted from the book's Ch8/Section81.ipynb
("Anomaly detection in timeseries of networks"). Tests the null hypothesis that the
latent positions (the RDPG parameters) are identical between windows t and t+1; small
p-values flag structural shifts. Corrected for multiple comparisons via Holm-Bonferroni
(implemented by hand to avoid a new dependency).

**Windows-specific caveat**: `latent_position_test` uses multiprocessing internally when
`workers=-1` (the book default). Jupyter under Windows may exhibit spawn-vs-fork semantics
issues. The smoke test below (5 pairs) should reveal hangs immediately; if so, fall back to
`workers=1` in the full run. Performance may differ significantly.

In [ ]:
def holm_bonferroni(
    pvalues: list[float], alpha: float = 0.05
) -> tuple[list[float], list[bool]]:
    """Holm-Bonferroni step-down correction, matching statsmodels' method="holm".

    Sorts p-values in ascending order by rank. For each rank i (0-indexed),
    the adjusted p-value is the running maximum of (m - i) * p[i] across
    all ranks up to i, capped at 1.0.

    Args:
        pvalues: List of p-values (length m).
        alpha: Significance threshold (default 0.05).

    Returns:
        (adjusted_pvalues, rejected): adjusted_pvalues is the list of
        Holm-adjusted p-values (same length, same order as input). rejected
        is a boolean list indicating which hypotheses are rejected at level alpha.
    """
    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.zeros(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adjusted_val = (m - rank) * pvalues[idx]
        running_max = max(running_max, adjusted_val)
        adjusted[idx] = min(running_max, 1.0)
    rejected = (adjusted < alpha).tolist()
    return adjusted.tolist(), rejected

In [ ]:
# Smoke test: 5 consecutive pairs of windows
print("Smoke test: latent_position_test on first 5 window transitions...")
window_ends_list = list(window_ends)
pvals_smoke = []
for t in range(min(5, len(window_ends_list) - 1)):
    adj_t = adj_tensor[t]
    adj_t1 = adj_tensor[t + 1]
    result = latent_position_test(
        adj_t1, adj_t, n_components=DHAT, n_bootstraps=100, workers=1
    )
    pvals_smoke.append(result.pvalue)
    print(f"  Window {t} -> {t+1}: p-value = {result.pvalue:.4f}")
print("✓ Smoke test complete (no hangs).")

## Chapter B — Full Run (can be slow; consider workers=1 if workers=-1 hangs)

In [ ]:
print(f"Running latent_position_test on {len(window_ends_list) - 1} window transitions...")
%time 
pvalues = [
    latent_position_test(
        adj_tensor[t + 1], adj_tensor[t], n_components=DHAT, n_bootstraps=200, workers=1
    ).pvalue
    for t in tqdm(range(len(window_ends_list) - 1), desc="latent_position_test")
]
print(f"Collected {len(pvalues)} p-values")

In [ ]:
adjusted_pvalues, rejected = holm_bonferroni(pvalues, alpha=0.05)
regime_df = pl.DataFrame({
    "transition": [f"{window_ends_list[t]} -> {window_ends_list[t+1]}" for t in range(len(pvalues))],
    "p_value": pvalues,
    "adjusted_p_value": adjusted_pvalues,
    "rejected": rejected
})
n_sig = sum(rejected)
print(f"Regime changes: {n_sig} / {len(pvalues)} transitions significant at α=0.05")
regime_df.head()

In [ ]:
# Plot (e): -log10(adjusted p-value) per transition, with significance threshold
regime_plot_df = pl.DataFrame({
    "transition_idx": list(range(len(pvalues))),
    "neg_log10_p": [-np.log10(p) if p > 0 else 20 for p in adjusted_pvalues],
    "significant": ["Yes" if r else "No" for r in rejected]
})

(
    ggplot(regime_plot_df, aes("transition_idx", "neg_log10_p", fill="significant"))
    + geom_bar(stat="identity", width=0.7)
    + geom_hline(yintercept=-np.log10(0.05), linetype="dashed", color="red", size=0.8)
    + scale_fill_manual(values={"Yes": "#d62728", "No": "#1f77b4"})
    + labs(
        x="Window Transition Index",
        y="−log₁₀(Holm-adjusted p)",
        title="Plot (e): Regime / Change-Point Detection (Latent Position Test)",
        fill="Significant"
    )
    + theme_minimal()
    + theme(figure_size=(12, 5))
)

## Chapter C: Dynamic Community Detection & Drift Tracking

Per-window community detection via Adjacency Spectral Embedding (ASE) + KMeans.
Each window's graph is embedded independently to a fixed latent dimension, then
clustered with automatic k-selection (silhouette score). Communities across windows
are compared via Adjusted Rand Index (ARI), which is label-permutation-invariant by
construction — **no Procrustes alignment or label remapping is used for the numeric
result**. The visual heatmap uses `remap_labels` purely as a labeling aid (for visual
continuity across time), explicitly flagged as non-rigorous.

In [ ]:
def compute_window_communities(
    adjacency: np.ndarray, *, max_clusters: int = 10,
    ase_n_components: int | None = None, random_state: int = 0
) -> np.ndarray:
    """ASE + graspologic KMeansCluster (auto-selects k via silhouette).

    Args:
        adjacency: Binary (n, n) adjacency matrix.
        max_clusters: Upper bound for k search (default 10).
        ase_n_components: Latent dimension for ASE (if None, uses sqrt(n)).
        random_state: Seed for reproducibility.

    Returns:
        labels: (n,) cluster assignments (0-indexed).
    """
    if ase_n_components is None:
        ase_n_components = max(2, int(np.sqrt(adjacency.shape[0])))
    embedding = AdjacencySpectralEmbed(n_components=ase_n_components).fit_transform(adjacency)
    kmeans = KMeansCluster(max_clusters=max_clusters, random_state=random_state)
    labels = kmeans.fit_predict(embedding)
    return labels

In [ ]:
# Smoke test: communities for the first 3 windows
print("Smoke test: community detection on first 3 windows...")
labels_smoke = []
for t in range(min(3, len(adj_tensor))):
    labels = compute_window_communities(adj_tensor[t], random_state=0)
    labels_smoke.append(labels)
    print(f"  Window {t}: {len(np.unique(labels))} communities detected")
print("✓ Smoke test complete.")

## Chapter C — Full Run: Community Detection

In [ ]:
print(f"Computing communities for {len(adj_tensor)} windows...")
%time 
labels_by_window = [
    compute_window_communities(adj_tensor[t], random_state=0)
    for t in tqdm(range(len(adj_tensor)), desc="communities")
]
cluster_counts = [len(np.unique(labels)) for labels in labels_by_window]
print(f"Community counts per window: min={min(cluster_counts)}, max={max(cluster_counts)}, mean={np.mean(cluster_counts):.1f}")

In [ ]:
# Compute ARI between consecutive windows
aris = [adjusted_rand_score(labels_by_window[t], labels_by_window[t + 1])
        for t in range(len(labels_by_window) - 1)]
ari_df = pl.DataFrame({
    "transition_idx": list(range(len(aris))),
    "ari": aris
})

# Plot (f): ARI over time
(
    ggplot(ari_df, aes("transition_idx", "ari"))
    + geom_line(size=0.8, color="#555555")
    + geom_point(size=2, color="#1f77b4")
    + labs(
        x="Window Transition Index",
        y="Adjusted Rand Index",
        title="Plot (f): Community Label Drift (ARI Between Consecutive Windows)"
    )
    + theme_minimal()
    + theme(figure_size=(10, 5))
)

In [ ]:
# Detailed comparison summary
print("\n" + "="*80)
print("METHOD COMPARISON SUMMARY")
print("="*80)

for method in methods:
    ks = results_all[method]["ks"]
    aris = ari_by_method[method]
    
    print(f"\n{method.upper():20s}")
    print(f"  k (communities):")
    print(f"    min={min(ks):2d}, max={max(ks):2d}, mean={np.mean(ks):5.1f}, std={np.std(ks):5.1f}")
    print(f"  ARI (label drift):")
    print(f"    min={min(aris):6.3f}, max={max(aris):6.3f}, mean={np.mean(aris):6.3f}, std={np.std(aris):6.3f}")

print("\nInterpretation:")
print("  - Methods with lower mean k: more conservative; fewer, larger communities.")
print("  - Methods with higher mean k: more granular; more, smaller communities.")
print("  - Methods with higher mean ARI: more stable labels across time (less churn).")
print("  - Methods with lower mean ARI: community memberships shift more frequently.")
print("="*80)

## Chapter C+ — Summary: Method Comparison

In [ ]:
# Plot (j): Optimal k evolution per method
k_evolution_data = []
for method in methods:
    for t_idx, k_val in enumerate(results_all[method]["ks"]):
        k_evolution_data.append({"method": method, "window_idx": t_idx, "k": k_val})

k_evolution_df = pl.DataFrame(k_evolution_data)

(
    ggplot(k_evolution_df, aes("window_idx", "k", color="method"))
    + geom_line(size=0.8)
    + geom_point(size=1.5)
    + labs(
        x="Window Index",
        y="Optimal k (number of communities)",
        color="Method",
        title="Plot (j): Optimal k Selection Over Time (All Methods)"
    )
    + theme_minimal()
    + theme(figure_size=(12, 6))
)

## Chapter C+ — Comparison: Optimal k Evolution Over Time

In [ ]:
# Compute ARI drift (between consecutive windows) for each method
ari_by_method = {}
for method in methods:
    aris = [adjusted_rand_score(results_all[method]["labels"][t], results_all[method]["labels"][t + 1])
            for t in range(len(results_all[method]["labels"]) - 1)]
    ari_by_method[method] = aris

# Plot (i): ARI drift comparison across methods
ari_comparison_data = []
for method in methods:
    for t_idx, ari_val in enumerate(ari_by_method[method]):
        ari_comparison_data.append({"method": method, "transition_idx": t_idx, "ari": ari_val})

ari_comparison_df = pl.DataFrame(ari_comparison_data)

(
    ggplot(ari_comparison_df, aes("transition_idx", "ari", color="method"))
    + geom_line(size=0.8)
    + facet_wrap("~method", ncol=3)
    + labs(
        x="Window Transition Index",
        y="Adjusted Rand Index",
        title="Plot (i): Community Label Drift (ARI) Across All Methods"
    )
    + theme_minimal()
    + theme(figure_size=(14, 6))
)

## Chapter C+ — Comparison: ARI Drift Per Method

In [ ]:
print(f"Running all {len(methods)} methods on {len(adj_tensor)} windows...")
results_all = {m: {"labels": [], "ks": [], "scores": []} for m in methods}

for method in methods:
    print(f"  Running {method}...")
    for t in tqdm(range(len(adj_tensor)), desc=method, leave=False):
        labels, k, score = compute_communities_with_method(adj_tensor[t], method=method, random_state=0)
        results_all[method]["labels"].append(labels)
        results_all[method]["ks"].append(k)
        results_all[method]["scores"].append(score)

print(f"\nCompleted all methods. Summary of k per method:")
for method in methods:
    ks = results_all[method]["ks"]
    print(f"  {method:20s}: min={min(ks):2d}, max={max(ks):2d}, mean={np.mean(ks):5.1f}")

## Chapter C+ — Full Run: All Methods Across All Windows

In [ ]:
methods = ["fixed", "silhouette", "modularity", "davies_bouldin", "calinski_harabasz"]
results_smoke = {m: [] for m in methods}

print("Smoke test: Comparing methods on first 3 windows...")
for t in range(min(3, len(adj_tensor))):
    print(f"  Window {t}:")
    for method in methods:
        labels, k, score = compute_communities_with_method(adj_tensor[t], method=method, random_state=0)
        results_smoke[method].append((k, score))
        print(f"    {method:20s}: k={k:2d}, score={score:8.4f}")
print("✓ Smoke test complete.")

## Chapter C+ — Smoke Test: Optimization Methods on First 3 Windows

In [ ]:
def compute_communities_with_method(adjacency, method="fixed", max_clusters=10, ase_n_components=None, random_state=0):
    """Compute communities using a specified optimization method.

    Args:
        adjacency: Binary (n, n) adjacency matrix.
        method: "fixed", "silhouette", "modularity", "davies_bouldin", or "calinski_harabasz".
        max_clusters: Upper bound for k search (used for all methods except fixed).
        ase_n_components: Latent dimension for ASE (if None, uses sqrt(n)).
        random_state: Seed for reproducibility.

    Returns:
        (labels, selected_k, score): cluster assignments, number of communities, and optimization score.
    """
    if ase_n_components is None:
        ase_n_components = max(2, int(np.sqrt(adjacency.shape[0])))
    embedding = AdjacencySpectralEmbed(n_components=ase_n_components, check_lcc=False).fit_transform(adjacency)

    if method == "fixed":
        kmeans = KMeansCluster(max_clusters=max_clusters, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
        selected_k = len(np.unique(labels))
        score = -1.0
    elif method == "silhouette":
        selected_k, score = _optimal_k_silhouette(embedding, max_clusters, random_state)
        kmeans = KMeansCluster(max_clusters=selected_k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
    elif method == "modularity":
        selected_k, score = _optimal_k_modularity(adjacency, embedding, max_clusters, random_state)
        kmeans = KMeansCluster(max_clusters=selected_k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
    elif method == "davies_bouldin":
        selected_k, score = _optimal_k_davies_bouldin(embedding, max_clusters, random_state)
        kmeans = KMeansCluster(max_clusters=selected_k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
    elif method == "calinski_harabasz":
        selected_k, score = _optimal_k_calinski_harabasz(embedding, max_clusters, random_state)
        kmeans = KMeansCluster(max_clusters=selected_k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
    else:
        raise ValueError(f"Unknown method: {method}")

    return labels, selected_k, score

In [ ]:
def _optimal_k_silhouette(embedding, max_clusters=10, random_state=0):
    """Find optimal k by maximizing silhouette coefficient in latent space."""
    best_k = 2
    best_score = -1.0
    for k in range(2, min(max_clusters + 1, len(embedding))):
        kmeans = KMeansCluster(max_clusters=k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
        if len(np.unique(labels)) < 2:
            continue
        score = silhouette_score(embedding, labels)
        if score > best_score:
            best_score = score
            best_k = k
    return best_k, best_score


def _optimal_k_modularity(adjacency, embedding, max_clusters=10, random_state=0):
    """Find optimal k by maximizing modularity in the original network."""
    G = nx.Graph(adjacency)
    best_k = 2
    best_mod = -1.0
    for k in range(2, min(max_clusters + 1, len(embedding))):
        kmeans = KMeansCluster(max_clusters=k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
        if len(np.unique(labels)) < 2:
            continue
        partition = [[] for _ in range(k)]
        for node_idx, label in enumerate(labels):
            partition[label].append(node_idx)
        partition = [p for p in partition if p]
        if len(partition) < 2:
            continue
        try:
            mod = nx.algorithms.community.modularity(G, partition)
            if mod > best_mod:
                best_mod = mod
                best_k = len(partition)
        except Exception:
            pass
    return best_k, best_mod


def _optimal_k_davies_bouldin(embedding, max_clusters=10, random_state=0):
    """Find optimal k by minimizing Davies-Bouldin index in latent space."""
    best_k = 2
    best_score = float("inf")
    for k in range(2, min(max_clusters + 1, len(embedding))):
        kmeans = KMeansCluster(max_clusters=k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
        if len(np.unique(labels)) < 2:
            continue
        score = davies_bouldin_score(embedding, labels)
        if score < best_score:
            best_score = score
            best_k = k
    return best_k, best_score


def _optimal_k_calinski_harabasz(embedding, max_clusters=10, random_state=0):
    """Find optimal k by maximizing Calinski-Harabasz index in latent space."""
    best_k = 2
    best_score = -1.0
    for k in range(2, min(max_clusters + 1, len(embedding))):
        kmeans = KMeansCluster(max_clusters=k, random_state=random_state)
        labels = kmeans.fit_predict(embedding)
        if len(np.unique(labels)) < 2:
            continue
        score = calinski_harabasz_score(embedding, labels)
        if score > best_score:
            best_score = score
            best_k = k
    return best_k, best_score

In [ ]:
# Build node × window community-membership heatmap
heatmap_data = []
for t, labels in enumerate(labels_by_window):
    for node_idx, node_name in enumerate(common_nodes):
        heatmap_data.append({"window_idx": t, "node": node_name, "community": labels[node_idx]})
heatmap_df = pl.DataFrame(heatmap_data).sort("window_idx").with_columns(
    pl.col("community").cast(pl.Utf8)
)

max_community_id = max(max(labels) for labels in labels_by_window)
print(f"Max community ID across all windows: {max_community_id}")
if max_community_id >= 6:
    print(f"⚠ Warning: {max_community_id + 1} communities detected; CATEGORICAL_6 has only 6 colors.")
    print("  Consider extending the palette or filtering to top communities.")

# Plot (g): Heatmap of community memberships
community_colors = CATEGORICAL_6[: min(6, max_community_id + 1)] + [
    "#cccccc"
] * max(0, max_community_id + 1 - 6)

(
    ggplot(heatmap_df.to_pandas(), aes("window_idx", "node", fill="community"))
    + geom_tile()
    + scale_fill_manual(values=community_colors)
    + labs(
        x="Window Index",
        y="Node (Stock)",
        fill="Community",
        title="Plot (g): Node Community Membership Over Time",
    )
    + theme_minimal()
    + theme(
        figure_size=(10, 8),
        axis_text_x=element_text(angle=45),
        axis_text_y=element_text(size=6),
    )
)

## Chapter C+: Community Detection Optimization Methods

The default **FIXED** method (Chapter C) uses `graspologic.cluster.KMeansCluster` with a hard
upper bound on k (`max_clusters`, typically 10). This treats all windows the same way and does
not optimize k based on the data. Four alternative **optimization strategies** determine k
independently per window, using only that window's data (no lookahead bias):

1. **SILHOUETTE** — Maximize average silhouette coefficient in latent (ASE) space.
   Balances cohesion and separation within clusters.

2. **MODULARITY** — Maximize modularity in the **original network** (graph-aware).
   Directly optimizes a classic community-detection objective.

3. **DAVIES_BOULDIN** — Minimize Davies-Bouldin index in latent space.
   Penalizes overlapping or poorly-separated clusters.

4. **CALINSKI_HARABASZ** — Maximize Calinski-Harabasz index in latent space.
   Ratio of between-cluster to within-cluster variance.

This chapter compares all five methods side-by-side on the same dataset, demonstrating:
- Optimal k per window (distribution and evolution over time)
- Adjusted Rand Index (ARI) drift under each method
- Community heatmaps for visual comparison

## Chapter D: Node Trajectory Embedding via Omnibus

Independent per-window ASE fits (Chapter C) produce *unrelated* coordinate systems
(would need Procrustes alignment to compare across time). In contrast,
`graspologic.embed.OmnibusEmbed` jointly embeds **all** windows at once into a single
shared, pre-aligned latent space. This allows direct node-trajectory visualization without
post-hoc alignment, revealing whether individual stocks drift, converge, or cluster
distinctly in latent space as market regimes shift.

In [ ]:
# Joint embedding of all windows
print(f"Running OmnibusEmbed on adjacency tensor of shape {adj_tensor.shape}...")
%time latent_tensor = OmnibusEmbed(n_components=DHAT, svd_seed=0).fit_transform(adj_tensor)
print(f"Latent tensor shape: {latent_tensor.shape}")
print(f"Expected: ({len(adj_tensor)}, {len(common_nodes)}, {DHAT})")

In [ ]:
def top_moving_nodes(
    latent_tensor: np.ndarray, node_names: list[str], k: int = 6
) -> list[str]:
    """Rank nodes by total latent-space displacement across windows.

    For each node, compute the sum of Euclidean distances between consecutive
    windows' latent positions, then return the k nodes with largest total displacement.

    Args:
        latent_tensor: (T, n, DHAT) latent positions across time.
        node_names: List of n node names, in the same order as latent_tensor axis 1.
        k: Number of top nodes to return (default 6, matching CATEGORICAL_6).

    Returns:
        top_k_names: List of k node names with largest total displacements.
    """
    total_displacements = np.zeros(latent_tensor.shape[1])
    for t in range(latent_tensor.shape[0] - 1):
        delta = latent_tensor[t + 1] - latent_tensor[t]
        total_displacements += np.linalg.norm(delta, axis=1)
    top_k_indices = np.argsort(-total_displacements)[:k]
    return [node_names[i] for i in top_k_indices]

In [ ]:
top_moving = top_moving_nodes(latent_tensor, common_nodes, k=6)
print(f"Top 6 most-moving nodes: {top_moving}")

# Build trajectory dataframe (each row is one node × window × latent coord pair)
traj_data = []
for t_idx, window_end in enumerate(window_ends):
    for n_idx, node_name in enumerate(common_nodes):
        if node_name in top_moving:
            lat = latent_tensor[t_idx, n_idx, :]
            traj_data.append({
                "window_end": window_end,
                "window_idx": t_idx,
                "node": node_name,
                "dim1": lat[0],
                "dim2": lat[1] if DHAT >= 2 else 0.0
            })
traj_df = pl.DataFrame(traj_data).sort(["node", "window_idx"])
print(f"Trajectory DataFrame: {traj_df.shape}")

In [ ]:
# Plot (h): 2D node trajectories in latent space
(
    ggplot(traj_df, aes("dim1", "dim2", color="node", group="node"))
    + geom_path(size=0.8, arrow=arrow(type="closed", length=0.1))
    + geom_point(
        data=traj_df.filter(pl.col("window_idx") == traj_df["window_idx"].max()),
        mapping=aes(fill="node"),
        size=3,
        stroke=0.5,
        color="black",
        alpha=0.8,
    )
    + scale_color_manual(values=CATEGORICAL_6[:len(top_moving)] +
                                 ["#cccccc"] * max(0, len(top_moving) - 6))
    + scale_fill_manual(values=CATEGORICAL_6[:len(top_moving)] +
                               ["#cccccc"] * max(0, len(top_moving) - 6))
    + labs(
        x=f"Latent Dim 1",
        y=f"Latent Dim 2",
        color="Node",
        fill="Node",
        title="Plot (h): Top 6 Node Trajectories in Omnibus Latent Space"
    )
    + theme_minimal()
    + theme(figure_size=(10, 8), legend_position="right")
)

## Final Reminder: Clear Outputs Before Committing
All of the cells in this extended notebook (original 32 + new 56) produce outputs that
must be cleared before any git commit. Use **Jupyter → Kernel → Restart & Clear All Outputs**
and save (⌘/Ctrl+S) before staging/committing.